<a href="https://colab.research.google.com/github/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_19_functions_first_class/note_lesson_19_functions_first_class.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Урок 19 — Функції як об'єкти першого класу: сервіс доставки росте

Карантин минув, а доставка «Смачно + Таксі» лишилася. Щотижня — нове: команда `drivers`, промокоди `SUMMER10`, `STUDENT15`, `NIGHT5`, сортування доставок, повідомлення в SMS і Telegram. У коді модуля 1 кожне таке прохання — ще один `elif` чи ще одна функція-близнюк.

Одна ідея розв'язує все: **функція — такий самий об'єкт, як число чи рядок**. Її можна покласти у змінну, у словник, передати в іншу функцію, повернути з функції.

План уроку:

- **Частина I** — вбудовані інструменти, що приймають функцію: `key=`, `lambda`, `map`, `filter`, `zip`, `any`, `all`, `reduce`. Вчимося ними користуватися.
- **Частина II** — патерни: словник команд, callback, фабрика, конвеєр, декоратор і головний принцип — чисте ядро, ефекти на краях. Тут функції вищого порядку пишемо самі.

Виконуй клітинки **зверху вниз**; перед **Прогнозом** спершу скажи, що буде. Теорія — у книзі: [Урок 19. Функції як об'єкти першого класу](https://nikoriakviktot.github.io/PY-Course-Victor-Nikoriak-22-09-2026/modules/m2/lesson_18/).

## 🔁 Пригадай (без підглядання)

1. Що отримує і що повертає декоратор з уроку 10?
2. Що робить `max(revenue, key=revenue.get)`?
3. Як `app.py` фінального проєкту обирав команду?
4. Що буде, якщо вдруге пройти циклом по тому самому генератору?

<details>
<summary>Відповіді</summary>

1. Отримує функцію, повертає нову функцію-обгортку.
2. Повертає ключ з найбільшим значенням: `max` викликає `revenue.get` для кожного ключа.
3. Ланцюжком `if command == ...`.
4. Нічого: генератор вичерпаний, другий цикл не зробить жодного кроку.

</details>

In [7]:
def discount(price, percent):
    return price * (100 - percent) / 100

In [8]:
discount(500, 10)

450.0

In [9]:
discount(800, 10)

720.0

In [11]:
from functools import partial

discount_10 = partial(discount, percent=10)

In [12]:
discount_10(500)

450.0

In [14]:
discount_10(700)

630.0

## 1. Функція — це об'єкт

**Прогноз:** що надрукують три рядки?

**Функція** — звичайне визначення через `def`.

In [17]:
def fee_for(district):
    """Вартість доставки за районом."""
    return {"Поділ": 60, "Оболонь": 80}.get(district, 100)

**Рядок 1 — тип.** Ім'я без дужок — сам об'єкт. Який у нього тип?

In [18]:
print(type(fee_for))

<class 'function'>


**Рядок 2 — ім'я.** У функції-об'єкта є атрибути, як у будь-якого об'єкта.

In [19]:
print(fee_for.__name__)

fee_for


**Рядок 3 — виклик.** З дужками — це вже результат виклику.

In [20]:
print(fee_for("Поділ"))

60


<details>
<summary>Відповідь</summary>

`<class 'function'>`, `fee_for`, `60`. Без дужок — сам об'єкт-функція, з дужками — результат виклику.

</details>

**Приклад 1. Функцію — в іншу змінну.** Нової функції не створюється: `delivery_fee` і `fee_for` — два імені одного об'єкта, тому `is` дає `True`.

In [21]:
delivery_fee = fee_for
print(delivery_fee("Оболонь"), delivery_fee is fee_for)

80 True


**Приклад 2. Функції — у список.** Проходимо циклом і викликаємо кожну з тими самими даними.

In [22]:
bills = [540.0, 320.0, 980.0]
for summary in [len, sum, max]:
    print(summary.__name__, summary(bills))

len 3
sum 1840.0
max 980.0


# Частина I. Вбудовані функції, що приймають функцію

## 2. Функції вищого порядку і функціональний підхід

Ти вже передавав функцію в іншу функцію: `max(revenue, key=revenue.get)`. Функцію, яка **приймає** або **повертає** іншу функцію, називають **функцією вищого порядку** (*higher-order function*).

У Python є готовий набір таких функцій. З нього починається **функціональний підхід**: замість циклу з `append`, прапорцями і лічильниками ти кажеш, *що* зробити з даними:

| Інструмент | Що робить | Що повертає |
|---|---|---|
| `sorted` / `min` / `max` з `key=f` | порівнює елементи за `f(елемент)` | список / один елемент |
| `lambda x: вираз` | коротка функція без імені, щоб одразу її передати | функцію |
| `map(f, дані)` | обчислює `f(x)` для кожного `x` | ледачий ітератор |
| `filter(f, дані)` | лишає ті `x`, для яких `f(x)` істинне | ледачий ітератор |
| `zip(a, b, …)` | зшиває елементи з однаковим номером у кортежі | ледачий ітератор |
| `any(дані)` / `all(дані)` | чи істинний хоч один / кожен елемент | `True` / `False` |
| `functools.reduce(f, дані, поч)` | згортає дані в одне значення своєю операцією `f` | одне значення |

Дані для всього розділу — доставки за зміну:

📖 Документація: [Functional Programming HOWTO](https://docs.python.org/3/howto/functional.html), [вбудовані функції](https://docs.python.org/3/library/functions.html).

**Тип запису.** `Delivery` — іменований кортеж з уроку 13: поля за іменем, змінити не можна.

In [23]:
from typing import NamedTuple


class Delivery(NamedTuple):
    order_id: int
    district: str
    fare: int
    driver: str

**Доставки за зміну** — список з чотирьох записів.

In [24]:
deliveries = [
    Delivery(1, "Оболонь", 230, "D-3"),
    Delivery(2, "Поділ", 180, "D-1"),
    Delivery(3, "Оболонь", 270, "D-3"),
    Delivery(4, "Печерськ", 150, "D-2"),
]
print(deliveries[0])

Delivery(order_id=1, district='Оболонь', fare=230, driver='D-3')


**Окремі списки** вартостей і районів — для прикладів, де потрібні лише числа чи рядки.

In [25]:
fares = [d.fare for d in deliveries]
districts = [d.district for d in deliveries]
print(fares)
print(districts)

[230, 180, 270, 150]
['Оболонь', 'Поділ', 'Оболонь', 'Печерськ']


### 2.1 `key=`: за чим порівнювати

**Задача:** відсортувати доставки від дешевшої до дорожчої.

`sorted` вміє порівнювати числа й рядки, але не знає, як порівнювати доставки. Дамо йому функцію-ключ: «для кожної доставки обчисли ось це число — і порівнюй числа». Щоб побачити, **коли** `sorted` її викликає, функція друкує кожен свій виклик.

In [26]:
def by_fare(delivery):
    print(f"  key(№{delivery.order_id}) -> {delivery.fare}")
    return delivery.fare


cheap_first = sorted(deliveries, key=by_fare)
print([d.order_id for d in cheap_first])

  key(№1) -> 230
  key(№2) -> 180
  key(№3) -> 270
  key(№4) -> 150
[4, 2, 1, 3]


**`max` з тим самим ключем.** Ключ знову обчислюється для кожного елемента, а повертається одна доставка — найдорожча.

In [27]:
print(max(deliveries, key=by_fare).order_id)

  key(№1) -> 230
  key(№2) -> 180
  key(№3) -> 270
  key(№4) -> 150
3


**`reverse=True`** — від більшого до меншого. Працює і без `key`, якщо елементи вже порівнюються (числа).

In [28]:
print(sorted(fares, reverse=True))

[270, 230, 180, 150]


Що видно з друку:

- `key=by_fare` — **без дужок**: передаємо саму функцію, викликає її `sorted`;
- ключ обчислюється **один раз для кожного елемента**, потім порівнюються ключі;
- повертаються **самі доставки**, а не ключі: ключ — лише правило порівняння;
- `min` і `max` приймають той самий `key=`; `reverse=True` перевертає порядок.

**Правило:** `key=` — функція одного аргументу, яка з елемента робить те, що вміє порівнюватися (число, рядок, кортеж).

📖 Документація: [`sorted`](https://docs.python.org/3/library/functions.html#sorted), [`max`](https://docs.python.org/3/library/functions.html#max), [Sorting HOWTO — key functions](https://docs.python.org/3/howto/sorting.html#key-functions).

#### Ключ — кортеж

Кортежі порівнюються поелементно: спершу перші елементи, при рівності — другі.

**Прогноз:** у якому порядку `sorted` поверне доставки з ключем «район, потім вартість»?

In [30]:
def by_district_then_fare(delivery):
    return (delivery.district, delivery.fare)


for d in sorted(deliveries, key=by_district_then_fare):
    print(d.district, d.fare)

Оболонь 230
Оболонь 270
Печерськ 150
Поділ 180


<details>
<summary>Відповідь</summary>

Оболонь 230, Оболонь 270, Печерськ 150, Поділ 180 — спершу за районом (за абеткою), у межах району — за вартістю.

</details>

### 2.2 `lambda`: функція в один вираз

Писати `def by_fare` заради одного `sorted` — довго. `lambda` створює ту саму функцію прямо там, де вона потрібна.

**Приклад 1. `def` і `lambda` — та сама функція.** Обидві отримують доставку й повертають її вартість.

In [31]:
def by_fare(delivery):
    return delivery.fare


fare_of = lambda delivery: delivery.fare      # те саме, але без def і return

print(by_fare(deliveries[0]), fare_of(deliveries[0]))

230 230


**Приклад 2. `lambda` — звичайний об'єкт `function`**, тільки без власного імені: `__name__` у всіх — `'<lambda>'`.

In [32]:
print(type(fare_of).__name__, fare_of.__name__)

function <lambda>


**Приклад 3. `lambda` прямо в `key=`** — так її пишуть найчастіше: без імені, у місці виклику.

In [33]:
print([d.order_id for d in sorted(deliveries, key=lambda d: d.fare)])
print(min(deliveries, key=lambda d: d.fare).driver)

[4, 2, 1, 3]
D-2


Як читати `lambda`:

```
lambda delivery: delivery.fare
       ────┬───  ──────┬──────
       параметри  вираз — його результат повертається автоматично
```

- Це звичайний об'єкт-функція (`function`), тільки з іменем `'<lambda>'`.
- Параметрів може бути скільки завгодно, зокрема зі значеннями за замовчуванням.
- Тіло — **рівно один вираз**. Не можна `return`, `if`-інструкцію, цикл, присвоєння. Умовний **вираз** `A if умова else B` — можна.

Різні види параметрів — кожен в окремому прикладі. Імена (`no_args`, `add`…) тут лише для того, щоб показати виклик; у реальному коді таку `lambda` одразу передають у `key=` чи `map`.

**Без параметрів.** Перед двокрапкою нічого; виклик — з порожніми дужками.

In [34]:
no_args = lambda: 60
print(no_args())

60


**Кілька параметрів** — через кому, як у `def`.

In [35]:
add = lambda a, b: a + b
print(add(2, 3))

5


**Значення за замовчуванням.** Не передали `fee` — береться 60; передали за іменем — береться передане.

In [36]:
with_fee = lambda bill, fee=60: bill + fee
print(with_fee(500), with_fee(500, fee=90))

560 590


**Умовний вираз** `A if умова else B` — це вираз, тому в `lambda` можна. Інструкцію `if …:` з двокрапкою — не можна.

In [ ]:
label = lambda fare: "дорого" if fare > 200 else "ок"
print([label(fare) for fare in fares])

**Правило:** `lambda` — для короткої функції, яку **одразу передаєш** (`key=`, `map`, `filter`). Якщо хочеш дати їй ім'я (`fare_of = lambda ...`), пиши `def`: буде нормальне `__name__`, докстрінг і зрозумілий traceback. Так радить і PEP 8.

**Перевір себе:** чи спрацює `lambda fare: return fare + 20`?

<details>
<summary>Відповідь</summary>

Ні, `SyntaxError`: `return` — інструкція, а тіло `lambda` — лише вираз. Правильно: `lambda fare: fare + 20`.

</details>

📖 Документація: [туторіал: Lambda Expressions](https://docs.python.org/3/tutorial/controlflow.html#lambda-expressions), [довідник: lambda](https://docs.python.org/3/reference/expressions.html#lambda), [PEP 8 про присвоєння lambda](https://peps.python.org/pep-0008/#programming-recommendations).

### 2.3 `map`: застосувати функцію до кожного елемента

**Задача:** до кожної вартості додати сервісний збір 20 грн. Спершу — знайомим циклом, потім — `map`.

**Циклом:** порожній список, `for`, `append`.

In [ ]:
with_fee = []
for fare in fares:
    with_fee.append(fare + 20)
print(with_fee)

**Через `map`:** те саме одним рядком. `list(...)` потрібен, бо `map` повертає не список (про це — далі).

In [ ]:
print(list(map(lambda fare: fare + 20, fares)))

`map(функція, дані)` викликає `функція(x)` для кожного `x` і віддає результати в тому самому порядку. Цикл, порожній список і `append` пише за тебе Python.

Але `map` повертає не список, а **ледачий ітератор** — як генератор з уроку 11.

**Прогноз:** у якому порядку з'являться рядки? Що надрукує останній `print`?

**Функція з друком** — щоб бачити кожен її виклик.

In [ ]:
def add_service(fare):
    print(f"  add_service({fare})")
    return fare + 20

**Створюємо `map`.** Чи викликалась уже `add_service`?

In [ ]:
lazy = map(add_service, fares)
print("map створено:", type(lazy).__name__)

**`next`** — просимо один елемент.

In [ ]:
print("перший:", next(lazy))

**`list`** — забираємо все, що лишилось.

In [ ]:
print("решта:", list(lazy))

**Ще раз `list`** з того самого `map`.

In [ ]:
print("ще раз:", list(lazy))

<details>
<summary>Відповідь</summary>

Коли `map` створено, `add_service` **ще жодного разу не викликалась**. `next` обчислює один елемент, `list` — усі, що лишились. Останній `print` дає `[]`: ітератор вичерпаний, як генератор.

Потрібен результат кілька разів — збережи список: `with_fee = list(map(...))`.

</details>

`lambda` не обов'язкова: у `map` можна передати будь-яку готову функцію — вбудовану, свою або метод класу (`str.upper` — це функція, яка приймає рядок).

**Метод класу як функція.** `str.upper("Поділ")` — те саме, що `"Поділ".upper()`, тож `str.upper` можна передати в `map`.

In [ ]:
print(list(map(str.upper, districts)))

**Вбудована функція.** `len` для кожного району — довжина назви.

In [ ]:
print(list(map(len, districts)))

**Наша функція** `fee_for` з розділу 1 — вартість доставки для кожного району.

In [ ]:
print(list(map(fee_for, districts)))

**Числа → рядки для `join`.** `join` склеює лише рядки, тому спершу `map(str, ...)`.

In [ ]:
print(" + ".join(map(str, fares)), "=", sum(fares))

`map` може взяти **кілька** послідовностей: тоді функція отримує по одному елементу з кожної — `f(a[0], b[0])`, `f(a[1], b[1])`, …

**Прогноз:** вартостей 4, чайових 3. Скільки сум буде?

**Дві послідовності:** функція отримує вартість і чайові з однаковим номером.

In [ ]:
tips = [20, 0, 30]
print(list(map(lambda fare, tip: fare + tip, fares, tips)))

**Вбудована функція двох аргументів:** `max` отримує пари `(1, 4)`, `(5, 2)`, `(3, 6)`.

In [ ]:
print(list(map(max, [1, 5, 3], [4, 2, 6])))

<details>
<summary>Відповідь</summary>

Три: `[250, 180, 300]`. `map` зупиняється на **найкоротшій** послідовності, четверта вартість мовчки пропадає. `max` отримує пари `(1, 4)`, `(5, 2)`, `(3, 6)` → `[4, 5, 6]`.

</details>

📖 Документація: [`map`](https://docs.python.org/3/library/functions.html#map), [ітератори](https://docs.python.org/3/glossary.html#term-iterator).

### 2.4 `filter`: лишити тільки потрібні

**Задача:** відібрати доставки від 200 грн.

`filter(функція, дані)` викликає функцію для кожного елемента і **лишає сам елемент**, якщо результат істинний. Таку функцію (повертає «так/ні») називають **предикатом**.

**Фільтр чисел:** предикат `fare >= 200` повертає `True` чи `False`.

In [ ]:
print(list(filter(lambda fare: fare >= 200, fares)))

**Фільтр об'єктів:** предикат дивиться на поле, а лишаються цілі доставки. `filter` теж ледачий — це об'єкт `filter`, а не список.

In [ ]:
expensive = filter(lambda d: d.fare >= 200, deliveries)
print(type(expensive).__name__)
print([d.order_id for d in expensive])

Різниця з `map`: `map` **змінює** кожен елемент (стільки ж результатів), `filter` **не змінює**, а відкидає частину (результатів стільки ж або менше).

Предикат може повертати не лише `True`/`False` — `filter` дивиться на істинність. `filter(None, дані)` лишає просто істинні елементи.

**Прогноз:** які коментарі лишать два фільтри?

**`filter(None, …)`** — лишає істинні рядки, тобто непорожні.

In [ ]:
comments = ["без цибулі", "", "дзвонити в домофон", "", "   "]
print(list(filter(None, comments)))

**`filter(str.strip, …)`** — предикат повертає рядок без пробілів по краях; порожній результат — хиба.

In [ ]:
print(list(filter(str.strip, comments)))

<details>
<summary>Відповідь</summary>

Перший лишає й `"   "`: рядок з пробілів не порожній, отже істинний. Другий його відкидає: `"   ".strip()` дає `""`, а це хиба.

</details>

📖 Документація: [`filter`](https://docs.python.org/3/library/functions.html#filter), [перевірка істинності](https://docs.python.org/3/library/stdtypes.html#truth-value-testing).

### 2.5 `map` і `filter` чи comprehension?

Те саме вміє list comprehension з уроку 6. Порівняймо: «збір 20 грн тільки для доставок від 200».

**Два записи одного й того самого:** `map` + `filter` проти comprehension.

In [ ]:
r1 = list(map(lambda fare: fare + 20, filter(lambda fare: fare >= 200, fares)))
r2 = [fare + 20 for fare in fares if fare >= 200]
print(r1, r2, r1 == r2)

**Генераторний вираз** — круглі дужки замість квадратних. Теж ледачий, як `map`; зручно передати одразу в `sum`.

In [ ]:
r3 = (fare + 20 for fare in fares if fare >= 200)
print(sum(r3))

Comprehension читається зліва направо, а `map(..., filter(...))` — зсередини назовні. Як обирати:

| Ситуація | Пиши |
|---|---|
| вираз пишеш прямо тут (`fare + 20`, `d.fare >= 200`) | comprehension |
| функція **вже є** й має ім'я | `map(str.upper, districts)`, `filter(str.strip, comments)` |
| потрібна лінивість | `map` / `filter` або генераторний вираз `( … )` |
| коротка функція для `key=` | `lambda` |

📖 Документація: [List Comprehensions](https://docs.python.org/3/tutorial/datastructures.html#list-comprehensions), [генераторні вирази](https://docs.python.org/3/howto/functional.html#generator-expressions-and-list-comprehensions).

### 2.6 `zip`: зшити кілька списків

**Задача:** номери замовлень, водії й оцінки клієнтів прийшли трьома окремими списками. Треба пройтися по них разом.

`zip(a, b, …)` бере по одному елементу з кожної послідовності з **однаковим номером** і складає їх у кортеж. Теж ледачий ітератор.

**Два списки → пари.** `zip` — ледачий ітератор, тому для друку загортаємо в `list`.

In [ ]:
order_ids = [1, 2, 3, 4]
drivers = ["D-3", "D-1", "D-3", "D-2"]
ratings = [5, 4, 5]

pairs = zip(order_ids, drivers)
print(type(pairs).__name__)
print(list(pairs))

**Три списки в циклі `for`.** Кортеж одразу розпаковуємо в три змінні.

In [ ]:
for order_id, driver, rating in zip(order_ids, drivers, ratings):
    print(f"№{order_id}: {driver}, оцінка {rating}")

Де замовлення №4? Оцінок лише три, і `zip`, як `map`, **мовчки зупиняється на найкоротшій**. Для звіту це помилка, яку легко не помітити. Якщо довжини мусять збігатися, додай `strict=True` (Python 3.10+): буде `ValueError`.

In [ ]:
try:
    list(zip(order_ids, ratings, strict=True))
except ValueError as error:
    print("ValueError:", error)

Три часті прийоми з `zip`:

**1. Словник з двох списків:** перший — ключі, другий — значення.

In [ ]:
fee_table = dict(zip(["Поділ", "Оболонь", "Печерськ"], [60, 80, 90]))
print(fee_table)

**2. Сусідні пари.** `fares` і `fares[1:]` зсунуті на один елемент, тож `zip` дає пари «попередня, наступна».

In [ ]:
print(list(zip(fares, fares[1:])))
print([b - a for a, b in zip(fares, fares[1:])])

**3. Розпакувати назад:** `zip(*рядки)` перетворює рядки на стовпці.

In [ ]:
rows = [(1, "D-3"), (2, "D-1"), (3, "D-3")]
ids, names = zip(*rows)
print(ids, names)

У третьому прийомі `*rows` розпаковує список в окремі аргументи (як `*args` з уроку 8): `zip((1, "D-3"), (2, "D-1"), (3, "D-3"))`. `zip` бере перші елементи → `(1, 2, 3)`, другі → `('D-3', 'D-1', 'D-3')`.

📖 Документація: [`zip`](https://docs.python.org/3/library/functions.html#zip), [розпакування аргументів `*`](https://docs.python.org/3/tutorial/controlflow.html#unpacking-argument-lists).

### 2.7 `any` і `all`: питання «так/ні» до всієї послідовності

**Задача:** чи є хоч одна доставка на Поділ? Чи всі доставки дорожчі за 100 грн?

- `any(дані)` — `True`, якщо **хоч один** елемент істинний;
- `all(дані)` — `True`, якщо **кожен** елемент істинний.

Усередину зазвичай пишуть генераторний вираз — без зайвих дужок.

**`any`:** чи є хоч одна доставка на Поділ?

In [ ]:
print(any(d.district == "Поділ" for d in deliveries))

**`all`:** чи всі доставки дорожчі за 100? А за 200?

In [ ]:
print(all(d.fare > 100 for d in deliveries))
print(all(d.fare > 200 for d in deliveries))

**Що робить `any` всередині** — те саме звичайним циклом.

In [ ]:
def has_district(deliveries, district):
    for d in deliveries:
        if d.district == district:
            return True
    return False


print(has_district(deliveries, "Поділ"))

Цикл `has_district` виходить, щойно знайшов відповідь. `any` і `all` роблять так само — це називають **скороченим обчисленням** (*short-circuit*).

**Прогноз:** скільки разів надрукується «перевіряю» в кожному рядку? `fares = [230, 180, 270, 150]`.

In [ ]:
def is_expensive(fare):
    print(f"  перевіряю {fare}")
    return fare > 200


print("any:", any(is_expensive(f) for f in fares))

**`all`** з тим самим предикатом:

In [ ]:
print("all:", all(is_expensive(f) for f in fares))

**`any` зі списком** — квадратні дужки замість генераторного виразу:

In [ ]:
print("any зі списком:", any([is_expensive(f) for f in fares]))

<details>
<summary>Відповідь</summary>

- `any` — **1** раз: 230 > 200, відповідь «так» уже відома.
- `all` — **2** рази: 180 не > 200, відповідь «ні» вже відома.
- `any([...])` — **4** рази: квадратні дужки спершу будують увесь список, а `any` отримує готові значення. Скорочення втрачено — пиши без `[ ]`.

</details>

**Прогноз:** що повернуть `any([])` і `all([])`?

In [ ]:
print(any([]), all([]))

**Не лише умови:** `any` / `all` дивляться на істинність самих елементів. Порожній рядок — хиба.

In [ ]:
print(all(["Поділ", "", "Оболонь"]))

**Пастка порожніх даних:** доставок ще не було, а перевірка «усі оплачені» проходить.

In [ ]:
shift = []
print("усі оплачені?", all(d.fare > 0 for d in shift))
print("усі оплачені, і зміна не порожня?", bool(shift) and all(d.fare > 0 for d in shift))

<details>
<summary>Відповідь</summary>

`False True`. `any([])` — немає жодного істинного. `all([])` — немає жодного **хибного**, тому «всі» формально правда.

Пастка: перевірка `all(...)` на порожніх даних завжди проходить. Якщо порожні дані — теж помилка, перевір це окремо: `shift and all(...)`.

</details>

📖 Документація: [`any`](https://docs.python.org/3/library/functions.html#any), [`all`](https://docs.python.org/3/library/functions.html#all).

### 2.8 `reduce`: згорнути послідовність в одне значення

`sum`, `max`, `any`, `all` — **згортки** (*reductions*): з багатьох значень роблять одне. Кожна знає лише свою операцію. Загальну згортку для **будь-якої** операції дає `functools.reduce(функція, дані, початкове)`.

Функція отримує два аргументи: **накопичене** значення й **наступний** елемент, і повертає нове накопичене. Подивимось кроки:

**Без початкового значення:** накопиченим стає перший елемент.

In [ ]:
from functools import reduce


def add_logged(total, fare):
    print(f"  {total} + {fare} = {total + fare}")
    return total + fare


print(reduce(add_logged, fares))

**З початковим значенням `0`** — на один крок більше.

In [ ]:
print(reduce(add_logged, fares, 0))

Без початкового значення перший елемент стає накопиченим: `230 + 180`, потім `410 + 270`, потім `680 + 150`. З початковим `0` — на один крок більше.

```
reduce(f, [a, b, c, d])  ==  f(f(f(a, b), c), d)
```

**Прогноз:** що буде з порожнім списком — з початковим значенням і без нього?

**Порожній список з початковим значенням:**

In [ ]:
print(reduce(lambda total, fare: total + fare, [], 0))

**Порожній список без початкового значення:**

In [ ]:
try:
    reduce(lambda total, fare: total + fare, [])
except TypeError as error:
    print("TypeError:", error)

<details>
<summary>Відповідь</summary>

З початковим значенням — повертає його (`0`). Без нього — `TypeError`: нема з чого почати. Тому якщо дані можуть бути порожні, **завжди** давай початкове значення.

</details>

**Правило:** для суми, максимуму, «хоч один», «всі» є готові `sum`, `max`, `any`, `all` — вони читаються краще. `reduce` — коли операція своя (наприклад, прогнати значення крізь список функцій — побачиш у розділі «Конвеєр»).

📖 Документація: [`functools.reduce`](https://docs.python.org/3/library/functools.html#functools.reduce), [Functional Programming HOWTO: functools](https://docs.python.org/3/howto/functional.html#the-functools-module).

**Своя операція:** найдорожча доставка через `reduce` — переможець кожної пари йде далі. Результат той самий, що в `max(..., key=...)`, але читається гірше — тут краще вбудована.

In [ ]:
top = reduce(lambda best, d: best if best.fare >= d.fare else d, deliveries)
print(top.order_id, max(deliveries, key=lambda d: d.fare).order_id)

## 🛠 Вправа 0. Звіт за зміну без циклів з `append`

Кожен пункт — один рядок з вбудованою функцією цього розділу. Цикл `for` з `append` не використовуй.

1. `fares_with_fee` — **список** вартостей зі збором 20 грн (`map` + `lambda`);
2. `obolon_ids` — номери доставок на Оболонь (`filter`, потім `map`);
3. `labels` — рядки `"№1 D-3"`, `"№2 D-1"`, … (`zip` двох списків);
4. `has_cheap` — чи є доставка дешевша за 160 грн; `all_valid` — чи кожен водій має позивний, що починається з `"D-"` (`any` / `all`);
5. `top_driver` — водій найдорожчої доставки (`max` з `key=lambda`).

In [ ]:
order_ids = [d.order_id for d in deliveries]
drivers = [d.driver for d in deliveries]

# YOUR CODE HERE
# BEGIN SOLUTION
fares_with_fee = list(map(lambda fare: fare + 20, fares))
obolon_ids = list(map(lambda d: d.order_id, filter(lambda d: d.district == "Оболонь", deliveries)))
labels = [f"№{order_id} {driver}" for order_id, driver in zip(order_ids, drivers)]
has_cheap = any(fare < 160 for fare in fares)
all_valid = all(driver.startswith("D-") for driver in drivers)
top_driver = max(deliveries, key=lambda d: d.fare).driver
# END SOLUTION

print(fares_with_fee, obolon_ids, labels, has_cheap, all_valid, top_driver, sep="\n")
assert fares_with_fee == [250, 200, 290, 170], "потрібен список: загорни map у list()"
assert obolon_ids == [1, 3]
assert labels == ["№1 D-3", "№2 D-1", "№3 D-3", "№4 D-2"]
assert has_cheap is True and all_valid is True
assert top_driver == "D-3"
print("✅ Вправа 0 пройдена")

# Частина II. Патерни: функції вищого порядку пишемо самі

Досі функцію приймали вбудовані `sorted`, `map`, `filter`, `reduce`. Тепер напишемо **свої** функції, що зберігають, приймають і повертають функції. Разом вони дають **функціональну архітектуру**: програма — це композиція функцій, які **перетворюють дані**, а не постійно змінюють спільний стан.

| Патерн | Проблема сервісу | Ідея | Розділ |
|---|---|---|---|
| Словник команд | `run` росте ланцюжком `if` | функції — значення словника | 3 |
| Callback | `dispatch` не має знати, SMS чи Telegram | «коли» вирішує функція, «що» — той, хто викликає | 4 |
| Фабрика + замикання | функції-близнюки `apply_summer10`, `apply_student15` | функція створює функцію, а та пам'ятає параметри | 5 |
| Конвеєр | ціна проходить кілька кроків | список функцій, вихід кроку — вхід наступного | 6 |
| `partial`, каррінг | фабрика заради одного зафіксованого аргументу | `partial(discount, 10)` — готова фабрика | 5 |
| Лінивий конвеєр | мільйон доставок не влазить у списки | генератори, `map`, `filter` — по одному елементу | 6 |
| Декоратор | додати лог чи перевірку, не змінюючи функцію | функція-обгортка з фабрики | 6 |
| Мемоїзація | повільний сервіс карт питають про той самий район | `@lru_cache` пам'ятає результати чистої функції | 6 |
| `on_success` / `on_error` | обробка помилок дублюється в кожному місці виклику | два callback-и: канал успіху й канал помилки | 7 |
| Чисте ядро | розрахунки змішані з `print` і SMS | обчислення — чисті функції, ефекти — на краях | 8 |

Кожен патерн покроково, «від проблеми до рішення», — у файлах поглиблення: [Callback](https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_19_functions_first_class/pattern_01_callback.md), [Factory](https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_19_functions_first_class/pattern_02_factory.md), [Decorator](https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_19_functions_first_class/pattern_03_decorator.md), [Pipeline](https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_19_functions_first_class/pattern_04_pipeline.md), [Pure Functions](https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_19_functions_first_class/pattern_05_pure_functions.md).

## 3. Словник команд замість ланцюжка `if`

Функцію можна покласти в словник, як будь-яке значення. Нова команда — функція і **один рядок** у словнику. `run` не змінюється.

**Команди, словник і `run`.** Значення словника — функції без дужок.

In [ ]:
def cmd_report(orders):
    return f"Замовлень: {len(orders)}, виторг: {sum(orders):.2f} грн"


def cmd_top(orders):
    return f"Найбільший чек: {max(orders):.2f} грн"


COMMANDS = {
    "report": cmd_report,
    "top": cmd_top,
}


def run(command, orders):
    handler = COMMANDS.get(command)
    if handler is None:
        return f"Помилка: невідома команда {command}. Є: {', '.join(COMMANDS)}"
    return handler(orders)

**Виклики:** дві відомі команди й одна невідома. Список доступних команд у підказці береться з того самого словника.

In [ ]:
orders = [540.0, 320.0, 980.0]
print(run("report", orders))
print(run("top", orders))
print(run("fly", orders))

## 🛠 Вправа 1. Команда `drivers`

Функція `cmd_drivers(deliveries)` повертає водіїв за виторгом від більшого до меншого: `"D-3 500, D-1 180, D-2 150"`. Додай її в `COMMANDS` — `run` не змінюй.

Підказка: виторг кожного водія зручно рахувати в `Counter`, а відсортувати імена — `sorted(..., key=totals.get, reverse=True)` (розділ 2.1).

In [ ]:
from collections import Counter


def cmd_drivers(deliveries):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    totals = Counter()
    for delivery in deliveries:
        totals[delivery.driver] += delivery.fare
    names = sorted(totals, key=totals.get, reverse=True)
    return ", ".join(f"{name} {totals[name]}" for name in names)
    # END SOLUTION


# YOUR CODE HERE — додай команду в COMMANDS
# BEGIN SOLUTION
COMMANDS["drivers"] = cmd_drivers
# END SOLUTION

print(run("drivers", deliveries))
assert run("drivers", deliveries) == "D-3 500, D-1 180, D-2 150"
assert "drivers" in run("fly", deliveries)
print("✅ Вправа 1 пройдена")

## 4. Callback: як повідомити — вирішує той, хто викликає

`map` і `filter` отримують функцію і викликають її самі. Так само може робити і наша функція: `dispatch` не знає, як повідомити клієнта, — вона викликає те, що їй передали.

**Функція, що приймає callback.** `dispatch` не знає, як саме повідомляти, — вона просто викликає `notify(текст)`.

In [ ]:
def dispatch(delivery, notify):
    """Відправляє водія і повідомляє клієнта через notify(текст)."""
    notify(f"Замовлення №{delivery.order_id}: водій {delivery.driver} виїхав, {delivery.fare} грн")

**Два канали — дві функції.** Передаємо їх без дужок; `dispatch` не змінюється.

In [ ]:
def notify_sms(text):
    print("SMS:", text)


def notify_telegram(text):
    print("Telegram:", text)


dispatch(deliveries[0], notify_sms)
dispatch(deliveries[1], notify_telegram)

**Callback у тестах.** `sent.append` — метод списку, теж функція. Замість SMS повідомлення потрапляє в список, і його можна перевірити.

In [ ]:
sent = []
dispatch(deliveries[2], sent.append)
print(sent)

## 5. Фабрика функцій і замикання

`make_discount(percent)` створює функцію-знижку, яка **пам'ятає** свій `percent`.

**Фабрика і дві створені нею функції.** Кожна пам'ятає свій `percent`.

In [ ]:
def make_discount(percent):
    """Фабрика: повертає функцію, що знижує чек на percent відсотків."""
    def apply(bill):
        return round(bill * (100 - percent) / 100, 2)
    return apply


summer10 = make_discount(10)
student15 = make_discount(15)
print(summer10(540.0), student15(540.0))

**Ім'я створеної функції** — `apply`: фабрика повертає свою внутрішню функцію.

In [ ]:
print(summer10.__name__)

**Реєстр промокодів** — словник, значення якого створила фабрика. Як таблиця команд.

In [ ]:
PROMO = {
    "SUMMER10": make_discount(10),
    "STUDENT15": make_discount(15),
    "NIGHT5": make_discount(5),
}

print(PROMO["NIGHT5"](980.0))

### `nonlocal`

**Прогноз:** що надрукує `print(other(), new_order_id())` після трьох викликів `new_order_id()`?

**Фабрика лічильників.** `count` живе в замиканні; `nonlocal` дозволяє його змінювати.

In [ ]:
def make_counter(start=0):
    count = start

    def next_id():
        nonlocal count
        count += 1
        return count

    return next_id

**Перший лічильник** — три виклики поспіль.

In [ ]:
new_order_id = make_counter()
print(new_order_id(), new_order_id(), new_order_id())

**Другий лічильник** з 100 — і ще один виклик першого.

In [ ]:
other = make_counter(100)
print(other(), new_order_id())

<details>
<summary>Відповідь</summary>

`101 4`: кожен виклик `make_counter` створює **свій** лічильник, вони не заважають один одному.

</details>

### Пастка: функції в циклі

**Прогноз:** що надрукує клітинка? Потім виправ через фабрику.

In [ ]:
fees = [lambda bill: bill + fee for fee in (60, 80, 100)]
print([f(500) for f in fees])

## 🛠 Вправа 2. Виправ пастку фабрикою

Напиши `make_fee_adder(fee)`, що повертає функцію `bill -> bill + fee`, і збери `fees` через неї.

In [ ]:
def make_fee_adder(fee):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    def add(bill):
        return bill + fee
    return add
    # END SOLUTION


fees = [make_fee_adder(fee) for fee in (60, 80, 100)]
print([f(500) for f in fees])
assert [f(500) for f in fees] == [560, 580, 600]
print("✅ Вправа 2 пройдена")

### `partial`: фабрика, яку не треба писати

Часто загальна функція вже є, а потрібна її «налаштована» версія з частиною зафіксованих аргументів. Це **часткове застосування** (*partial application*). Писати для цього фабрику не обов'язково: `functools.partial(функція, аргументи…)` повертає нову функцію, у якій ці аргументи вже підставлені.

**`partial` замість фабрики.** Результат той самий, що в `make_discount(10)`. `partial` пам'ятає функцію (`.func`) і зафіксовані аргументи (`.args`).

In [ ]:
from functools import partial


def discount(percent, bill):
    return round(bill * (100 - percent) / 100, 2)


summer10_p = partial(discount, 10)            # percent = 10 зафіксовано
print(summer10_p(540.0), make_discount(10)(540.0))
print(summer10_p.func.__name__, summer10_p.args)

**Реєстр промокодів через `partial`.**

In [ ]:
PROMO_P = {"SUMMER10": partial(discount, 10), "NIGHT5": partial(discount, 5)}
print(PROMO_P["NIGHT5"](980.0))

**Фіксуємо іменовані аргументи** вбудованої функції: «сортування від дорожчої» як окрема функція.

In [ ]:
expensive_first = partial(sorted, key=lambda d: d.fare, reverse=True)
print([d.order_id for d in expensive_first(deliveries)])

Позиційні аргументи `partial` підставляє **зліва**: `partial(discount, 10)(540.0)` — це `discount(10, 540.0)`.

**Прогноз:** що повернуть два рядки нижче?

**Позиційно:**

In [ ]:
print(partial(discount, 540.0)(10))

**За іменем:**

In [ ]:
print(partial(discount, bill=540.0)(10))

<details>
<summary>Відповідь</summary>

`-44.0` і `486.0`. У першому `540.0` став **першим** аргументом, тобто `percent`, і знижка вийшла 540 %. Щоб зафіксувати не перший параметр, передай його за іменем: `bill=540.0`.

</details>

**Каррінг** (*currying*) — запис функції кількох аргументів як ланцюжка функцій **одного** аргументу: `make_discount(10)(540.0)`. Наша фабрика — це вже каррінг вручну. Python не робить його автоматично; на практиці вистачає `partial`.

**Як обирати:** загальна функція вже є, треба лише зафіксувати аргументи — `partial`. Потрібна додаткова логіка під час створення (перевірка району, як у `make_fee`) чи стан (`nonlocal`) — фабрика.

📖 Документація: [`functools.partial`](https://docs.python.org/3/library/functools.html#functools.partial).

## 6. Конвеєр з функцій

Список функцій, які по черзі обробляють значення. **Прогноз:** чому два конвеєри з тими самими кроками дають різну ціну?

**Кроки й виконавець.** `run_pipeline` по черзі передає значення кожному кроку.

In [ ]:
def add_fee(bill):
    return bill + 80


def to_uah(bill):
    return round(bill)


def run_pipeline(value, steps):
    for step in steps:
        value = step(value)
    return value


checkout = [PROMO["SUMMER10"], add_fee, to_uah]
print(run_pipeline(540.0, checkout))

**Ті самі кроки, інший порядок:** спершу доставка, потім знижка.

In [ ]:
print(run_pipeline(540.0, [add_fee, PROMO["SUMMER10"], to_uah]))

<details>
<summary>Відповідь</summary>

`566` і `558`: у другому знижку отримала й доставка. Порядок кроків — бізнес-правило.

</details>

`run_pipeline` — це згортка з розділу 2.8: накопичене значення — ціна, елементи — кроки, операція — «застосуй крок».

In [ ]:
print(reduce(lambda value, step: step(value), checkout, 540.0))

### Лінивий конвеєр: дані йдуть по одному

`run_pipeline` проводить через кроки **одне** значення. А якщо даних мільйон — доставки з файлу чи БД? Збирати їх у список, потім ще один список після фільтра, потім ще один після `map` — дорого.

`map`, `filter` і генератори ліниві (розділ 2.3), тож з них складається **конвеєр для потоку**: кожен елемент проходить усі кроки, і лише тоді береться наступний. Обгорнемо кроки так, щоб вони друкували свої виклики.

**Прогноз:** в якому порядку з'являться рядки — спершу всі `filter`, потім усі `map`?

**Обгортка з друком** — ще одна маленька фабрика: показує кожен виклик кроку.

In [ ]:
def trace(name, fn):
    def wrapped(x):
        print(f"  {name}({x})")
        return fn(x)
    return wrapped

**Збираємо конвеєр:** потік → `filter` → `map`. Нічого ще не надрукується.

In [ ]:
stream = iter(fares)                     # як рядки файлу: приходять по одному
expensive = filter(trace("filter", lambda fare: fare >= 200), stream)
with_fee = map(trace("map", lambda fare: fare + 20), expensive)
print("конвеєр зібрано — ще нічого не пораховано")

**Запускаємо** — беремо результати циклом.

In [ ]:
for total in with_fee:
    print("отримав", total)

<details>
<summary>Відповідь</summary>

Ні, вони чергуються: `filter(230)` → `map(230)` → `отримав 250`, далі `filter(180)` (відкинуто), `filter(270)` → `map(270)` → … Кожен елемент проходить увесь конвеєр, і лише тоді береться наступний. Проміжних списків немає.

</details>

Що це дає на великих даних:

**Пам'ять:** готовий список з мільйона чисел проти `map` над тими самими даними.

In [ ]:
import sys

big = range(1_000_000)
print("список:", sys.getsizeof([x * 2 for x in big]), "байт")
print("map:   ", sys.getsizeof(map(lambda x: x * 2, big)), "байт")

**Потік доставок + `islice`:** генератор видає доставки по одній; `islice` бере перші три дорогі й зупиняє конвеєр. Лічильник `created` показує, скільки доставок генератор встиг створити.

In [ ]:
from itertools import islice

created = 0


def delivery_stream(n):
    """Генератор: мільйон доставок, але в пам'яті — завжди одна."""
    global created
    for i in range(n):
        created += 1
        yield Delivery(i, "Поділ", 100 + i % 300, "D-1")


top3 = islice(filter(lambda d: d.fare >= 390, delivery_stream(1_000_000)), 3)
print([d.order_id for d in top3], "— створено доставок:", created)

Список займає мегабайти, а `map` — кілька десятків байтів: він зберігає не результати, а «рецепт». `islice(дані, 3)` бере перші три елементи й зупиняє весь конвеєр: з мільйона доставок створено лише 293.

**Правило:** коли даних багато або вони приходять потоком — збирай конвеєр з генераторів, `map`, `filter`, `islice` і перетворюй у список лише в самому кінці (або взагалі ні: `sum`, `for`, запис у файл).

📖 Документація: [`itertools.islice`](https://docs.python.org/3/library/itertools.html#itertools.islice), [генератори](https://docs.python.org/3/howto/functional.html#generators).

### Кроки мають бути чистими

Конвеєр працює, лише якщо кожен крок **чистий**.

**Прогноз:** що буде в `cart` після кожного виклику?

**Чиста версія** повертає новий список.

In [ ]:
def add_packaging(items):
    return items + ["пакет"]


cart = ["борщ", "вареники"]
print(add_packaging(cart), cart)

**Нечиста версія** дописує в той самий список.

In [ ]:
def add_packaging_bad(items):
    items.append("пакет")
    return items


print(add_packaging_bad(cart), cart)

<details>
<summary>Відповідь</summary>

Чиста `add_packaging` повертає новий список і не змінює `cart`. Нечиста дописує «пакет» у сам `cart`.

</details>

**Чиста функція** (*pure function*) — це дві умови разом:

1. **Той самий вхід → той самий результат.** Результат залежить лише від аргументів, а не від часу, глобальних змінних чи файлів.
2. **Немає побічних ефектів** (*side effects*): не змінює аргументи, глобальні змінні, не друкує, не пише у файл чи БД, не надсилає SMS.

Чистоту можна перевірити: виклич двічі з тими самими аргументами — результати однакові, а навколо нічого не змінилось.

**Прогноз:** які з функцій нижче чисті? Що надрукують два виклики `add_to_shift(230)`?

**1. `add_to_shift`** — змінює глобальну змінну.

In [ ]:
shift_total = 0


def add_to_shift(fare):
    global shift_total
    shift_total += fare
    return shift_total


print(add_to_shift(230), add_to_shift(230))

**2. `fee_logged`** — рахує й друкує.

In [ ]:
def fee_logged(district):
    print("  рахую доставку:", district)
    return fee_for(district)


print(fee_logged("Поділ"))

**3. `night_fee`** — читає годинник. Результат залежить від того, коли ти запускаєш клітинку.

In [ ]:
from datetime import datetime


def night_fee(district):
    return fee_for(district) * (2 if datetime.now().hour >= 22 else 1)


print(night_fee("Поділ"), "— зараз", datetime.now().hour, "год")

**4. `surge_fee`** — той самий розрахунок, але годину передають аргументом.

In [ ]:
def surge_fee(district, hour):
    return fee_for(district) * (2 if hour >= 22 else 1)


print(surge_fee("Поділ", 23), surge_fee("Поділ", 23), surge_fee("Поділ", 12))

<details>
<summary>Відповідь</summary>

Чиста лише `surge_fee`.

- `add_to_shift` — `230 460`: той самий аргумент, різні результати, бо функція змінює `shift_total`.
- `fee_logged` — результат правильний, але функція ще й друкує. Сто викликів — сто рядків у консолі.
- `night_fee` — результат залежить від того, **коли** її викликали; тест, що пройшов удень, упаде вночі.
- `surge_fee` — той самий код, але годину передають аргументом. Тестувати легко: `assert surge_fee("Поділ", 23) == 120`.

</details>

Прийом з `surge_fee` універсальний: **прихований вхід зроби аргументом**.

#### Незмінність: не змінюй — створи нове

Найпростіше не зламати дані — взяти тип, який змінити **не можна**. `Delivery` — `NamedTuple`, тобто кортеж. Щоб «змінити» вартість, створюємо нову доставку через `_replace`, а стара лишається як була.

**«Змінити» — означає створити нову:** `_replace` повертає нову доставку, стара не змінилась.

In [ ]:
d = deliveries[0]
discounted = d._replace(fare=d.fare - 30)
print(d.fare, discounted.fare, d is discounted)

**Спроба змінити поле напряму** — помилка.

In [ ]:
try:
    d.fare = 0
except AttributeError:
    print("AttributeError: NamedTuple змінити не можна")

Навіщо це все:

- **передбачуваність** — щоб зрозуміти результат, досить подивитися на аргументи;
- **тести** — `assert f(x) == y`, без підготовки глобального стану й без перехоплення `print`;
- **порядок і повтор** — чисті кроки можна переставляти, повторювати, кешувати результат;
- **паралельність** — дві чисті функції не заважають одна одній, бо не мають спільного стану.

📖 Поглиблення: [Pure Functions & Immutability](https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_19_functions_first_class/pattern_05_pure_functions.md).

### Декоратор — це теж фабрика

Декоратор з уроку 10 тепер складається з відомих частин: функція, яка **приймає** функцію (вищого порядку), **створює** нову (фабрика) і **пам'ятає** оригінал (замикання). Сама функція при цьому не змінюється.

**Декоратор `log_calls`:** повертає обгортку, що викликає оригінал і друкує аргументи й результат.

In [ ]:
from functools import wraps


def log_calls(func):
    @wraps(func)
    def wrapper(*args):
        result = func(*args)
        print(f"  {func.__name__}{args} -> {result}")
        return result
    return wrapper

**Без `@`** — як звичайна фабрика, до кожного кроку конвеєра. Самі кроки не змінились.

In [ ]:
logged_checkout = [log_calls(step) for step in checkout]
print(run_pipeline(540.0, logged_checkout))

**Із `@`** — те саме, що `add_tip = log_calls(add_tip)`.

In [ ]:
@log_calls
def add_tip(bill):
    return bill + 50


add_tip(566)
print(add_tip.__name__)

`checkout` лишився чистим: лог додано **ззовні**, обгорткою. Потрібен лог — обгорни, не потрібен — бери оригінальні кроки. `@wraps(func)` копіює ім'я і докстрінг, тому `add_tip.__name__` — `add_tip`, а не `wrapper`.

📖 Поглиблення: [Decorator](https://github.com/NikoriakViktot/PY-Course-Victor-Nikoriak-22-09-2026/blob/main/module_2/lessons/lesson_19_functions_first_class/pattern_03_decorator.md); документація: [`functools.wraps`](https://docs.python.org/3/library/functools.html#functools.wraps).

### Мемоїзація: декоратор, що пам'ятає результати

Чиста функція для тих самих аргументів **завжди** повертає те саме. Отже, результат можна запам'ятати й удруге не рахувати. Це **мемоїзація** (*memoization*), а готовий декоратор для неї — `functools.lru_cache`.

Нехай час доїзду в район рахує повільний сервіс карт. `print` усередині лише показує, коли функція **справді** працює.

**Прогноз:** скільки разів надрукується «рахую маршрут» на чотири доставки?

**Функція з кешем.** `@lru_cache` над визначенням — і все.

In [ ]:
from functools import lru_cache


@lru_cache(maxsize=128)
def route_minutes(district):
    """Нібито повільний запит до сервісу карт."""
    print(f"  рахую маршрут до {district}...")
    return {"Поділ": 15, "Оболонь": 25, "Печерськ": 20}.get(district, 40)

**Чотири доставки** — чотири запити часу доїзду.

In [ ]:
for d in deliveries:
    print(f"№{d.order_id} {d.district}: {route_minutes(d.district)} хв")

**Статистика кешу:** скільки разів влучили й скільки рахували.

In [ ]:
print(route_minutes.cache_info())

<details>
<summary>Відповідь</summary>

Тричі: Оболонь, Поділ, Печерськ. Для доставки №3 Оболонь уже в кеші — функція не викликалась. `cache_info()`: `hits=1` (влучання в кеш), `misses=3` (рахували). `maxsize=128` — скільки результатів пам'ятати; найдавніше використані витісняються (*least recently used*).

</details>

Зверни увагу: при влучанні в кеш `print` усередині **не виконався**. Тому кешувати можна лише **чисті** функції:

- `night_fee` з `@lru_cache` назавжди запам'ятає денний тариф — годинник більше не читатиметься;
- функція, що надсилає SMS, з кешем надішле його лише раз.

Аргументи мають бути **хешованими** (як ключі словника): кеш — це словник «аргументи → результат».

**Список як аргумент** — не хешований:

In [ ]:
try:
    route_minutes(["Поділ", "Оболонь"])
except TypeError as error:
    print("TypeError:", error)

**Кортеж** хешований — працює (невідомий «район» → 40).

In [ ]:
print(route_minutes(("Поділ", "Оболонь")))

**Правило:** `@lru_cache` — лише на чисті функції з хешованими аргументами. Без обмеження розміру — `@functools.cache` (те саме, що `lru_cache(maxsize=None)`).

📖 Документація: [`functools.lru_cache`](https://docs.python.org/3/library/functools.html#functools.lru_cache), [`functools.cache`](https://docs.python.org/3/library/functools.html#functools.cache).

## 7. Розібраний приклад: оформлення замовлення

Зберемо все разом: реєстр промокодів, вартість доставки за районом, конвеєр. Крок за кроком — кожна частина окремо.

**Крок 1. Тарифи — дані, а не `if`.** Вартість доставки в кожен район — словник.

In [ ]:
FEES = {"Поділ": 60, "Оболонь": 80, "Печерськ": 90}

**Крок 2. Фабрика кроку доставки.** Район перевіряється **один раз**, під час створення кроку. Функція `add` пам'ятає `fee` через замикання.

In [ ]:
def make_fee(district):
    """Фабрика: крок конвеєра, що додає вартість доставки в район."""
    if district not in FEES:
        raise ValueError(f"не доставляємо в район {district}")
    fee = FEES[district]

    def add(bill):
        return bill + fee
    return add


to_obolon = make_fee("Оболонь")
print(to_obolon(486.0))

**Крок 2а. Невідомий район** — помилка вже при створенні кроку, ще до будь-яких обчислень.

In [ ]:
try:
    make_fee("Троєщина")
except ValueError as error:
    print("ValueError:", error)

**Крок 3. Список кроків для замовлення.** Знижка — лише якщо є промокод; промокод береться з реєстру `PROMO`, доставка — з фабрики, округлення — готова `to_uah`. Дивимось, які функції потрапили в список.

In [ ]:
def checkout_steps(promo_code, district):
    """Кроки для замовлення: знижка (якщо є промокод) → доставка → гривні."""
    steps = []
    if promo_code:
        if promo_code not in PROMO:
            raise ValueError(f"промокод {promo_code} не дійсний")
        steps.append(PROMO[promo_code])
    steps.append(make_fee(district))
    steps.append(to_uah)
    return steps


print([step.__name__ for step in checkout_steps("SUMMER10", "Оболонь")])
print([step.__name__ for step in checkout_steps(None, "Поділ")])

**Крок 4. `checkout`** — лише запускає конвеєр. Він не знає, які саме кроки виконує: новий крок — рядок у `checkout_steps`, а не переписаний `checkout`.

In [ ]:
def checkout(bill, promo_code, district):
    return run_pipeline(bill, checkout_steps(promo_code, district))


print(checkout(540.0, "SUMMER10", "Оболонь"))

**Крок 5. Усі випадки разом:** три коректні замовлення, хибний промокод і район, куди не доставляємо. Помилки — `ValueError` з поясненням.

In [ ]:
CHECKOUT_CASES = [(540.0, "SUMMER10", "Оболонь"), (320.0, None, "Поділ"),
                  (980.0, "STUDENT15", "Печерськ"), (540.0, "FREE100", "Поділ"),
                  (540.0, None, "Троєщина")]

for bill, code, district in CHECKOUT_CASES:
    try:
        print(bill, code, district, "->", checkout(bill, code, district))
    except ValueError as error:
        print(bill, code, district, "-> помилка:", error)

### Два канали: `on_success` і `on_error`

У циклі вище `try/except` і `print` живуть у коді, що викликає `checkout`. Кожне нове місце виклику (бот, сайт, звіт) повторить цю обробку. Передамо **дві** функції: одну для успіху, другу для помилки. Обчислення лише вирішує, **в який канал** віддати результат, а що з ним робити — вирішує той, хто викликає. Це callback з розділу 4, тільки на дві колії; підхід називають *railway-oriented programming*.

**`place_order`** віддає результат в один із двох каналів.

In [ ]:
def place_order(bill, promo_code, district, on_success, on_error):
    """Рахує замовлення і віддає результат в один із двох каналів."""
    try:
        total = checkout(bill, promo_code, district)
    except ValueError as error:
        on_error(f"{district}, {promo_code}: {error}")
    else:
        on_success(total)

**Канали — списки:** зручно для звіту й тестів.

In [ ]:
accepted, rejected = [], []
for bill, code, district in CHECKOUT_CASES:
    place_order(bill, code, district, accepted.append, rejected.append)
print("прийнято:", accepted)
print("відхилено:", rejected)

**Канали — друк:** ті самі замовлення, інша обробка, `place_order` не змінився.

In [ ]:
place_order(540.0, "SUMMER10", "Оболонь",
            on_success=lambda total: print("✅ до сплати", total),
            on_error=lambda message: print("❌", message))
place_order(540.0, None, "Троєщина",
            on_success=lambda total: print("✅ до сплати", total),
            on_error=lambda message: print("❌", message))

Ті самі замовлення — два різні способи обробки, а `place_order` не змінювався: у першому випадку колбеки збирають результати у списки (зручно для звіту й тестів), у другому — друкують.

> ⚠️ **Лови лише очікувані помилки.** `except ValueError`, а не `except Exception`. `ValueError` — це «замовлення некоректне» (район, промокод), і його законне місце — канал `on_error`. А `TypeError` чи `NameError` — баг у коді. Якби `place_order` ловив усе, баг тихо став би «відхиленим замовленням».

## 8. Чисте ядро, ефекти на краях

Головний принцип функціональної архітектури: **відокремити обчислення від побічних ефектів**.

```
   край (вхід)             чисте ядро                     край (вихід)
 input, файл, БД  ──►  checkout, PROMO, make_fee,   ──►  print, SMS, Telegram, БД
                       shift_messages
                       лише чисті функції,
                       перевіряються assert
```

**Задача:** наприкінці зміни кожен водій отримує повідомлення з виторгом. Ось як це пишуть «в лоб»:

**Змішана функція:** рахує виторг і тут же «надсилає» SMS.

In [ ]:
from collections import Counter


def close_shift_mixed(deliveries):
    totals = Counter()
    for d in deliveries:
        totals[d.driver] += d.fare
    for driver in sorted(totals):
        print(f"SMS {driver}: за зміну {totals[driver]} грн")

**Виклик** — результат можна лише побачити, а не перевірити.

In [ ]:
close_shift_mixed(deliveries)

Працює, але:

- **перевірити** суми можна, лише перехопивши `print`;
- для Telegram доведеться **скопіювати** весь підрахунок;
- підрахунок і надсилання — одна функція: помилка в будь-якій частині ламає обидві.

Розділимо: **ядро** лише рахує й повертає тексти, **край** лише надсилає. Надсилання — callback з розділу 4.

**Ядро** рахує й повертає тексти. Перевіряємо його звичайними `assert` — без SMS і без `print`.

In [ ]:
def shift_messages(deliveries):
    """Чисте ядро: доставки -> тексти повідомлень. Нічого не надсилає."""
    totals = Counter()
    for d in deliveries:
        totals[d.driver] += d.fare
    return [f"{driver}: за зміну {totals[driver]} грн" for driver in sorted(totals)]


assert shift_messages(deliveries) == [
    "D-1: за зміну 180 грн", "D-2: за зміну 150 грн", "D-3: за зміну 500 грн",
]
assert shift_messages([]) == []
print(shift_messages(deliveries))

**Край** лише передає тексти в callback. Канал змінюємо аргументом.

In [ ]:
def close_shift(deliveries, notify):
    """Край: тут і тільки тут побічний ефект."""
    for text in shift_messages(deliveries):
        notify(text)


close_shift(deliveries, notify_sms)
close_shift(deliveries[:1], notify_telegram)

`shift_messages` усередині змінює `totals` — але це **її власна** локальна змінна, створена в цьому виклику. Ззовні цього не видно, тому функція чиста: чистота — про те, що бачить той, хто викликає.

**Правило:** якщо функція і рахує, і щось робить зі світом — розріж її. Обчислення — у чисту функцію, що повертає дані. Ефект — у маленьку функцію на краю, яка лише передає ці дані далі. Так побудований і наш `checkout`: він повертає ціну, а не друкує її.

## 🛠 Вправа 3. Винеси чисте ядро

`remind_unpaid_mixed` і шукає неоплачені замовлення, і друкує нагадування. Розріж її на дві:

- `unpaid_reminders(orders)` — **чиста**: повертає список текстів `"Нагадування: замовлення №2 не оплачене"`;
- `remind_unpaid(orders, notify)` — **край**: викликає `notify` для кожного тексту.

`orders` — список пар `(order_id, paid)`.

In [ ]:
def remind_unpaid_mixed(orders):
    for order_id, paid in orders:
        if not paid:
            print(f"Нагадування: замовлення №{order_id} не оплачене")


def unpaid_reminders(orders):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    return [f"Нагадування: замовлення №{order_id} не оплачене"
            for order_id, paid in orders if not paid]
    # END SOLUTION


def remind_unpaid(orders, notify):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    for text in unpaid_reminders(orders):
        notify(text)
    # END SOLUTION


orders_paid = [(1, True), (2, False), (3, False), (4, True)]
assert unpaid_reminders(orders_paid) == [
    "Нагадування: замовлення №2 не оплачене",
    "Нагадування: замовлення №3 не оплачене",
]
assert unpaid_reminders([]) == []
assert orders_paid == [(1, True), (2, False), (3, False), (4, True)], "ядро не змінює вхід"

sent = []
remind_unpaid(orders_paid, sent.append)
assert sent == unpaid_reminders(orders_paid)
remind_unpaid(orders_paid, notify_sms)
print("✅ Вправа 3 пройдена")

## 🛠 Вправа 4. Промокод з обмеженням

`make_limited_discount(percent, uses)` — як `make_discount`, але після `uses` застосувань `apply` піднімає `ValueError("промокод вичерпано")`. Лічильник — у замиканні, через `nonlocal`.

In [ ]:
def make_limited_discount(percent, uses):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    left = uses

    def apply(bill):
        nonlocal left
        if left == 0:
            raise ValueError("промокод вичерпано")
        left -= 1
        return round(bill * (100 - percent) / 100, 2)
    return apply
    # END SOLUTION


lucky = make_limited_discount(20, 3)
assert [lucky(500.0), lucky(500.0), lucky(250.0)] == [400.0, 400.0, 200.0]
try:
    lucky(500.0)
except ValueError as error:
    assert str(error) == "промокод вичерпано"
else:
    raise AssertionError("четвертий виклик мав підняти ValueError")
other = make_limited_discount(20, 1)
assert other(100.0) == 80.0, "окремий промокод — окремий лічильник"
PROMO["LUCKY20"] = make_limited_discount(20, 3)
assert checkout(540.0, "LUCKY20", "Поділ") == 492
print("✅ Вправа 4 пройдена")

## 🐛 Вправа 5. Знайди помилку

Кожна клітинка — справжня помилка початківця. Розкоментуй зламаний рядок, подивись, що буде, і виправ.

In [ ]:
# Баг 1: sum = sum([540.0, 320.0])   ← розкоментуй разом з наступним рядком
# total_fares = sum([230, 180])
# BEGIN SOLUTION
total_bills = sum([540.0, 320.0])
total_fares = sum([230, 180])
# END SOLUTION
assert (total_bills, total_fares) == (860.0, 410)
print("✅ Баг 1: не називай змінні іменами вбудованих функцій")

In [ ]:
code_text = "summer10"
is_upper = code_text.isupper       # ← чому це «правда» для будь-якого рядка?
# BEGIN SOLUTION
is_upper = code_text.isupper()
# END SOLUTION
assert is_upper is False
print("✅ Баг 2: функція без дужок — об'єкт, і він завжди істинний")

In [ ]:
districts = ["Оболонь", "Поділ", "Печерськ"]
# print(sorted(districts, key=len(districts)))   ← TypeError: 'int' object is not callable
# BEGIN SOLUTION
result = sorted(districts, key=len)
# END SOLUTION
assert result == ["Поділ", "Оболонь", "Печерськ"]
print("✅ Баг 3: key= — функція, а не її результат")

## 🚀 Міні-проєкт: конвеєр для коментаря до замовлення

Клієнти пишуть коментарі до замовлень як завгодно: `"   без ЦИБУЛІ!!!   дзвонити   в домофон "`. Кухні й водієві потрібен акуратний текст. Кожен крок — чиста функція `str -> str`, конвеєр — список функцій, `run_pipeline` — той самий, що вище.

1. `strip_spaces` — прибирає пробіли по краях і зводить підряд кілька пробілів до одного;
2. `calm_down` — `"!!!"` → `"!"` (кілька знаків оклику — один);
3. `sentence_case` — перша літера велика, решта — малі;
4. ⭐ `make_limit(n)` — фабрика: крок, що обрізає текст до `n` символів і додає `"…"`, якщо обрізав.

In [ ]:
def strip_spaces(text):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    return " ".join(text.split())
    # END SOLUTION


def calm_down(text):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    while "!!" in text:
        text = text.replace("!!", "!")
    return text
    # END SOLUTION


def sentence_case(text):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    return text[:1].upper() + text[1:].lower()
    # END SOLUTION


def make_limit(n):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    def limit(text):
        return text if len(text) <= n else text[:n] + "…"
    return limit
    # END SOLUTION


comment_pipeline = [strip_spaces, calm_down, sentence_case, make_limit(30)]
raw = "   без ЦИБУЛІ!!!   дзвонити   в домофон "
clean = run_pipeline(raw, comment_pipeline)
print(repr(clean))
assert run_pipeline(raw, comment_pipeline[:3]) == "Без цибулі! дзвонити в домофон"
assert clean == "Без цибулі! дзвонити в домофон"
assert make_limit(10)("Без цибулі! дзвонити") == "Без цибулі…"
assert run_pipeline("", comment_pipeline) == ""
print("✅ Міні-проєкт пройдено")

## ✅ Самоперевірка

1. Чим `fee_for` відрізняється від `fee_for("Поділ")`?
2. Що повертає `sorted(deliveries, key=lambda d: d.fare)` — доставки чи вартості?
3. Чим `map` відрізняється від `filter`? Чому `list(m)` удруге дає `[]`?
4. Скільки пар дасть `zip([1, 2, 3], ["a", "b"])` і як не пропустити різницю довжин?
5. Що повертає `all([])` і чим це небезпечно?
6. Навіщо `reduce` початкове значення?
7. Що дає словник команд порівняно з ланцюжком `if`?
8. Чому `summer10` пам'ятає `percent`, хоча `make_discount` вже завершилась?
9. Навіщо `nonlocal`?
10. Чому порядок кроків у конвеєрі змінює ціну?
11. Назви дві умови чистої функції. Чим `night_fee` нечиста і як її виправити?
12. Що таке «чисте ядро, ефекти на краях» і навіщо так ділити код?

13. Що поверне `partial(discount, 540.0)(10)` і чому? Як зафіксувати `bill`?
14. Чому лінивий конвеєр не створює проміжних списків? Що робить `islice`?
15. Чому `@lru_cache` не можна ставити на `night_fee` чи на функцію, що надсилає SMS?
16. Навіщо `place_order` два callback-и і чому він ловить лише `ValueError`?

<details>
<summary>Відповіді</summary>

1. Об'єкт-функція проти результату виклику.
2. Доставки; ключ — лише правило порівняння.
3. `map` перетворює кожен елемент, `filter` відкидає частину. Обидва — ледачі ітератори, після першого проходу вичерпані.
4. Дві. `zip(..., strict=True)` підніме `ValueError`.
5. `True`: перевірка на порожніх даних завжди проходить — перевіряй порожнечу окремо.
6. Без нього порожній список дає `TypeError`; з ним — повертається початкове значення.
7. Нова команда — рядок у словнику; `run` не змінюється.
8. Це замикання: функція зберігає змінні оточення.
9. Щоб **змінити** змінну оточуючої функції; без нього присвоєння створює локальну.
10. Кожен крок працює з результатом попереднього.
11. Той самий вхід → той самий результат, і жодних побічних ефектів. `night_fee` читає годинник — прихований вхід; годину треба передати аргументом (`surge_fee`).
12. Обчислення — чисті функції, що повертають дані; `print`, SMS, БД — у маленьких функціях на краю. Ядро тестується `assert`, канал виводу легко замінити.
13. `-44.0`: `540.0` підставився першим, як `percent`. Треба `partial(discount, bill=540.0)`.
14. Кожен елемент проходить усі кроки, лише тоді береться наступний. `islice` бере перші n і зупиняє весь конвеєр.
15. Кеш не викликає функцію вдруге: прихований вхід (годинник) більше не читається, а побічний ефект (SMS) не повторюється.
16. Обчислення не знає, що робити з результатом: це вирішує той, хто викликає. `ValueError` — некоректне замовлення, інші помилки — баги, їх не можна ховати.

</details>

### Шпаргалка

```python
# Частина I — вбудовані
sorted(items, key=lambda d: d.fare)          # ключ: за чим порівнювати
max(items, key=lambda d: d.fare)             # min / max — так само
list(map(lambda x: x + 20, xs))              # перетворити кожен (ледачий → list)
list(map(str.upper, names))                  # готова функція — без lambda
list(filter(lambda x: x >= 200, xs))         # лишити, де предикат істинний
list(filter(None, xs))                       # лишити істинні
for a, b in zip(xs, ys, strict=True): ...    # зшити; strict — довжини мусять збігатися
dict(zip(keys, values))                      # словник з двох списків
any(x > 200 for x in xs)                     # хоч один (без [ ] — зупиняється рано)
all(x > 0 for x in xs)                       # кожен; all([]) == True!
reduce(lambda acc, x: acc + x, xs, 0)        # своя згортка; початкове — завжди

# Частина II — патерни
handler = COMMANDS.get(name)                 # словник команд
dispatch(delivery, notify_sms)               # callback — без дужок!

def make_discount(percent):                  # фабрика + замикання
    def apply(bill):
        return bill * (100 - percent) / 100
    return apply

def make_counter():
    count = 0
    def next_id():
        nonlocal count                       # змінити змінну оточення
        count += 1
        return count
    return next_id

summer10 = partial(discount, 10)             # часткове застосування
partial(discount, bill=540.0)                # не перший — за іменем

for step in steps:                           # конвеєр
    value = step(value)

logged = [log_calls(step) for step in steps]  # декоратор = фабрика-обгортка
lazy = map(f, filter(p, stream))             # лінивий конвеєр, по одному
first3 = islice(lazy, 3)                     # взяти 3 й зупинитись

@lru_cache(maxsize=128)                      # мемоїзація — лише чисті функції
def route_minutes(district): ...

place_order(..., on_success=ok.append, on_error=bad.append)   # два канали

def shift_messages(deliveries): ...          # чисте ядро: повертає дані
def close_shift(deliveries, notify): ...     # край: тільки тут ефекти
```

## Далі

**Урок 20 — Класи, простір імен.** Лічильник `make_counter` — дані й функція разом. Клас зробить це явно.

Поглиблення — п'ять патернів у файлах `pattern_01_callback.md` … `pattern_05_pure_functions.md` у теці уроку.

# ➕ Додаткові завдання: від системи до коду — «Таксі»

У теці уроку є гра-тренажер **«Від системи до коду»** (`Від системи до коду.html`): сім станцій, на яких сервіс доставки проєктується **зверху вниз** — спершу схема, потім функції, і лише наприкінці код. Відкрий її в браузері й пройди, якщо ще не пройшов.

Тепер друга половина бізнесу — **«Таксі»**: розрахунок поїздки. Ті самі сім питань, нова система. Не копіюй код доставки, а проходь питання по черзі: кожне завдання — одна станція.

| # | Питання перед кодом | Рівень | Інструмент |
|---|---|---|---|
| 1 | Що входить у систему і що з неї виходить? | Система | контракт · чорний ящик |
| 2 | З яких послідовних кроків складається перетворення? | Потік | конвеєр · список функцій |
| 3 | Що тут лише обчислює, а що торкається світу? | Функції | чисте ядро · краї |
| 4 | Які правила змінюються без зміни коду? | Функції | конфігурація · `partial` / фабрика |
| 5 | Яку поведінку треба підключати ззовні? | Функції | callback · гачок |
| 6 | Яке правило стосується багатьох функцій одразу? | Функції | декоратор · обгортка |
| 7 | Як усе з'єднати і запустити? | Код | runner + словник |

**Бізнес-правила таксі:**

- тариф — **15 грн за км**;
- уночі (з 22:00 до 06:00) кілометри множаться на **1.5**;
- промокод дає знижку у відсотках — **лише на кілометри**;
- подача авто — **40 грн**, фіксовано: без коефіцієнта і без знижки;
- сума округлюється до копійок.

Поїздка — словник, як замовлення в грі: `{"km": 12, "hour": 23, "promo": "NIGHT10", "driver": "D-3"}`. Кожен крок повертає **новий** словник з ключем `"total"`: `{**ride, "total": ...}`.

In [ ]:
# Службова клітинка: перевірка відповідей без підказки правильних (як у квесті уроку 5).
import hashlib

_SALT = "taxi-19:"


def _h(key, value):
    return hashlib.sha256((_SALT + key + "|" + str(value)).encode("utf-8")).hexdigest()[:12]


def check_answers(answers, expected_hashes, allowed):
    """Друкує ✅ / ❌ для кожного пункту; повертає кількість помилок."""
    wrong = 0
    for key, expected in expected_hashes.items():
        value = answers.get(key)
        if value not in allowed:
            print(f"⬜ {key}: ще не розкладено (можна: {', '.join(allowed)})")
            wrong += 1
        elif _h(key, value) == expected:
            print(f"✅ {key}: {value}")
        else:
            print(f"❌ {key}: не «{value}»")
            wrong += 1
    return wrong

## 🧭 Завдання 1 · Система: чорний ящик поїздки

Перш ніж писати хоч одну функцію, домовся про **контракт**: що система отримує, що віддає і за якими правилами працює всередині. Як саме вона це робить — поки неважливо.

```
  вхід  ──►  [ finish_ride ]  ──►  вихід
              правила всередині
```

**Місія:** для кожного пункту заміни `None` на `"вхід"`, `"правило"` (живе всередині ящика), `"вихід"` або `"зайве"` (не стосується цієї системи).

Підказки-питання: це приходить ззовні під час кожного виклику? Це однакове для всіх поїздок і визначає розрахунок? Це система віддає назовні? Розрахунок узагалі про це знає?

In [ ]:
CONTRACT = {
    "відстань поїздки, км": None,
    "година замовлення": None,
    "промокод клієнта": None,
    "тариф 15 грн за км": None,
    "нічний коефіцієнт ×1.5": None,
    "сума до сплати": None,
    "повідомлення водієві": None,
    "колір іконки таксі на карті": None,
}

# YOUR CODE HERE — заміни None на "вхід" / "правило" / "вихід" / "зайве"
# BEGIN SOLUTION
CONTRACT.update({
    "відстань поїздки, км": "вхід",
    "година замовлення": "вхід",
    "промокод клієнта": "вхід",
    "тариф 15 грн за км": "правило",
    "нічний коефіцієнт ×1.5": "правило",
    "сума до сплати": "вихід",
    "повідомлення водієві": "вихід",
    "колір іконки таксі на карті": "зайве",
})
# END SOLUTION

CONTRACT_HASHES = {
    "відстань поїздки, км": "bb855e4e995e",
    "година замовлення": "c509897a37ae",
    "промокод клієнта": "9297fcc53daa",
    "тариф 15 грн за км": "b1d1cec55db5",
    "нічний коефіцієнт ×1.5": "85ef0f994762",
    "сума до сплати": "7146fc76d71c",
    "повідомлення водієві": "f203ba3f0236",
    "колір іконки таксі на карті": "aa25a989d0dc"
}

errors = check_answers(CONTRACT, CONTRACT_HASHES, ["вхід", "правило", "вихід", "зайве"])
assert errors == 0, f"помилок: {errors} — переглянь рядки з ❌"
print("✅ Завдання 1: контракт готовий — поїздка → finish_ride → рахунок")

<details>
<summary>Чому так</summary>

- Відстань, година, промокод — **вхід**: свої для кожної поїздки.
- Тариф і нічний коефіцієнт — **правила**: однакові для всіх поїздок, живуть усередині.
- Сума — головний **вихід**. Повідомлення водієві — теж вихід, але назовні: це побічний ефект, він піде на край (завдання 3, 5).
- Колір іконки — інша система (інтерфейс). Розрахунок про неї не знає.

</details>

## 🧭 Завдання 2 · Потік: розклади ящик на станції конвеєра

Усередині ящика поїздка проходить кілька кроків. Кожен крок отримує поїздку й повертає нову; вихід одного — вхід наступного. **Порядок кроків — теж бізнес-логіка.**

Кроки вже написані (клітинка нижче). Прочитай їх: кожен робить лише свою роботу.

In [ ]:
from functools import partial

# КОНФІГУРАЦІЯ: правила, які змінюються без зміни коду
TARIFF_PER_KM = 15
NIGHT_FACTOR = 1.5
PICKUP_FEE = 40
PROMOS = {"NIGHT10": 10}

# ДАНІ: приходять під час виконання
ride = {"km": 12, "hour": 23, "promo": "NIGHT10", "driver": "D-3"}

In [ ]:
def validate_ride(ride):
    if ride["km"] <= 0:
        raise ValueError("поїздка без кілометрів")
    return ride


def km_fare(ride, per_km=TARIFF_PER_KM):
    return {**ride, "total": ride["km"] * per_km}


def night_rate(ride, factor=NIGHT_FACTOR):
    is_night_ride = ride["hour"] >= 22 or ride["hour"] < 6
    return {**ride, "total": ride["total"] * factor} if is_night_ride else ride


def apply_promo(ride, promos):
    code = ride.get("promo")
    if not code:
        return ride
    if code not in promos:
        raise ValueError(f"невідомий промокод: {code}")
    return {**ride, "total": ride["total"] * (100 - promos[code]) / 100}


def add_pickup(ride, fee=PICKUP_FEE):
    return {**ride, "total": ride["total"] + fee}


def round_total(ride):
    return {**ride, "total": round(ride["total"], 2)}

**Місія:** збери список `RIDE` з шести кроків у правильному порядку. Ціль для `ride` вище — **283.0 грн**.

- промокод бери з довідника через `partial(apply_promo, promos=PROMOS)` — так у список потрапляє функція одного аргументу;
- `run_pipeline` з розділу 6 уже є — він той самий для будь-яких даних.

Спершу порахуй на папері: 12 км × 15 = ? → уночі × 1.5 = ? → −10 % = ? → + подача = ?

In [ ]:
# YOUR CODE HERE — RIDE = [...]
# BEGIN SOLUTION
RIDE = [
    validate_ride,
    km_fare,
    night_rate,
    partial(apply_promo, promos=PROMOS),
    add_pickup,
    round_total,
]
# END SOLUTION

assert len(RIDE) == 6, "кроків має бути шість"
assert RIDE[0] is validate_ride, "перевірка — першою: некоректна поїздка не має доїхати до розрахунку"
assert RIDE[1] is km_fare, "решта кроків працює з ride['total'], а його створює km_fare"
assert RIDE[-1] is round_total, "округлення — останнім, інакше наступні кроки дадуть «довгі» копійки"

bill = run_pipeline(ride, RIDE)
print(bill)
assert bill["total"] == 283.0, f"вийшло {bill['total']}: чи не зачепили коефіцієнт або знижка подачу авто?"
assert ride == {"km": 12, "hour": 23, "promo": "NIGHT10", "driver": "D-3"}, "кроки не мають змінювати вхідну поїздку"
print("✅ Завдання 2: конвеєр зібрано — 283.0 грн")

<details>
<summary>Розбір</summary>

180 → 270 (ніч) → 243 (−10 %) → 283 (+40) → 283.0. Якщо поставити `add_pickup` перед `night_rate`, коефіцієнт зачепить подачу: (180 + 40) × 1.5 × 0.9 = 297.0. Кроки знають лише свою роботу, а порядок записаний окремо — у списку `RIDE`.

</details>

## 🧭 Завдання 3 · Функції: чисте ядро, брудні краї

Конвеєр надійний, лише коли його кроки **чисті**: той самий вхід — той самий вихід, і зовні нічого не змінюється. Файли, мережа, годинник і `print` теж потрібні, але живуть **на краях** — перед конвеєром або після нього.

**Місія (а):** розклади функції: `"ядро"` чи `"край"`.

Питання до кожної: чи змінює вона щось зовні? Чи може той самий виклик дати інший результат?

In [ ]:
CORE_OR_EDGE = {
    "km_fare(ride)": None,
    "add_pickup(ride)": None,
    "partial(apply_promo, promos=PROMOS)": None,
    "send_to_driver(bill)  # print / мережа": None,
    "read_rides('rides.csv')": None,
    "now_hour()  # datetime.now().hour": None,
    "add_stop(route, stop) з route.append(stop)": None,
}

# YOUR CODE HERE — заміни None на "ядро" / "край"
# BEGIN SOLUTION
CORE_OR_EDGE.update({
    "km_fare(ride)": "ядро",
    "add_pickup(ride)": "ядро",
    "partial(apply_promo, promos=PROMOS)": "ядро",
    "send_to_driver(bill)  # print / мережа": "край",
    "read_rides('rides.csv')": "край",
    "now_hour()  # datetime.now().hour": "край",
    "add_stop(route, stop) з route.append(stop)": "край",
})
# END SOLUTION

CORE_HASHES = {
    "km_fare(ride)": "cb2970042a5e",
    "add_pickup(ride)": "b20992421589",
    "partial(apply_promo, promos=PROMOS)": "173f484c7508",
    "send_to_driver(bill)  # print / мережа": "c6f9f62dbdc8",
    "read_rides('rides.csv')": "1750d56bf63b",
    "now_hour()  # datetime.now().hour": "786931917524",
    "add_stop(route, stop) з route.append(stop)": "9b7c47994298"
}

errors = check_answers(CORE_OR_EDGE, CORE_HASHES, ["ядро", "край"])
assert errors == 0, f"помилок: {errors}"
print("✅ Завдання 3а: межу проведено")

**Місія (б):** перенеси два «брудні» місця в ядро.

1. `now_hour()` читає годинник — це край. Напиши **чисту** `is_night(hour)`: годину читаємо на краю й передаємо аргументом. Ніч — з 22:00 до 06:00.
2. `add_stop_bad` дописує зупинку в чужий список. Напиши **чисту** `add_stop(route, stop)`, що повертає новий маршрут.

In [ ]:
from datetime import datetime


def now_hour():                       # край: залежить від того, коли викликали
    return datetime.now().hour


def add_stop_bad(route, stop):        # край: мутує аргумент
    route.append(stop)
    return route


def is_night(hour):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    return hour >= 22 or hour < 6
    # END SOLUTION


def add_stop(route, stop):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    return route + [stop]
    # END SOLUTION


assert [is_night(h) for h in (21, 22, 23, 0, 5, 6, 12)] == [False, True, True, True, True, False, False]
route = ["Поділ"]
assert add_stop(route, "Вокзал") == ["Поділ", "Вокзал"]
assert route == ["Поділ"], "чиста функція не змінює вхідний список"
print("зараз ніч?", is_night(now_hour()))   # край викликає ядро — і лише тут читаємо годинник
print("✅ Завдання 3б: годинник і мутація — на краях")

## 🧭 Завдання 4 · Функції: довідник і фабрика

Промокоди відрізняються лише відсотком, а коефіцієнти — лише числом. Отже, це **дані** (конфігурація), а не функції-близнюки `apply_night10`, `apply_taxi15`… Зв'язати дані з функцією можна через `partial` або через **фабрику** із замиканням.

**Місія:**

1. Додай у довідник `PROMOS` промокоди `TAXI15` (15 %) і `FIRST20` (20 %). Функції при цьому **не змінюються**.
2. Напиши фабрику `make_multiplier(factor)` — повертає крок, що множить `total` на `factor`.
3. Напиши фабрику `make_promo_step(promos)` — повертає крок, що застосовує `apply_promo` з довідником із замикання. Вона робить те саме, що `partial(apply_promo, promos=promos)`.

In [ ]:
# YOUR CODE HERE — 1. довідник
# BEGIN SOLUTION
PROMOS["TAXI15"] = 15
PROMOS["FIRST20"] = 20
# END SOLUTION


def make_multiplier(factor):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    def step(ride):
        return {**ride, "total": ride["total"] * factor}
    return step
    # END SOLUTION


def make_promo_step(promos):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    def step(ride):
        return apply_promo(ride, promos)
    return step
    # END SOLUTION


night = make_multiplier(NIGHT_FACTOR)
rain = make_multiplier(1.2)
trip = {"km": 10, "total": 100, "promo": "FIRST20"}

assert night(trip)["total"] == 150.0 and rain(trip)["total"] == 120.0
assert trip["total"] == 100, "крок повертає новий словник, а не змінює старий"
assert make_promo_step(PROMOS)(trip)["total"] == 80.0
assert make_promo_step(PROMOS)(trip) == partial(apply_promo, promos=PROMOS)(trip)
assert make_promo_step(PROMOS)({**trip, "promo": "TAXI15"})["total"] == 85.0
assert night.__closure__[0].cell_contents == 1.5, "factor має жити в замиканні"
print(sorted(PROMOS))
print("✅ Завдання 4: новий промокод — новий рядок даних, код не змінився")

## 🧭 Завдання 5 · Функції: гачок для повідомлень

`finish_ride` знає, **коли** треба повідомити (коли рахунок готовий), але не знає, **як**: водієві в застосунок, клієнтові SMS, у журнал. Тому вона залишає **гачок** — параметр `notify`, у який ззовні підключають функцію.

**Місія:**

1. `finish_ride(ride, notify)` — рахує `bill = run_pipeline(ride, RIDE)`, викликає `notify(bill)` і повертає `bill`;
2. два канали: `notify_driver(bill)` друкує `"Водію D-3: 12 км, 283.0 грн"`, `notify_client(bill)` — `"Клієнту: до сплати 283.0 грн"`.

In [ ]:
def finish_ride(ride, notify):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    bill = run_pipeline(ride, RIDE)
    notify(bill)
    return bill
    # END SOLUTION


def notify_driver(bill):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    print(f"Водію {bill['driver']}: {bill['km']} км, {bill['total']} грн")
    # END SOLUTION


def notify_client(bill):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    print(f"Клієнту: до сплати {bill['total']} грн")
    # END SOLUTION


finish_ride(ride, notify_driver)
finish_ride(ride, notify_client)

sent = []
assert finish_ride(ride, sent.append)["total"] == 283.0
assert len(sent) == 1 and sent[0]["driver"] == "D-3", "notify викликається рівно раз, з готовим рахунком"
print("✅ Завдання 5: два канали, finish_ride не змінювався")

**Прогноз:** що станеться з `finish_ride(ride, notify=notify_driver())`?

<details>
<summary>Відповідь</summary>

`notify_driver()` викликається **одразу**, ще до `finish_ride`, і падає з `TypeError`: їй не передали `bill`. А якби функція не мала параметрів, вона повернула б `None`, і всередині `finish_ride` впало б `TypeError: 'NoneType' object is not callable`. Гачку потрібна сама функція — **без дужок**.

</details>

## 🧭 Завдання 6 · Функції: охоронці навколо функції

Змінювати тариф може лише **менеджер**, і кожну зміну треба записати в журнал. Ці правила стосуються багатьох функцій (тариф, коефіцієнт, промокоди), тому їх не вписують усередину, а **обгортають** функцію шарами ззовні.

`log_call` уже готовий. **Місія:** напиши фабрику декораторів `requires_role(role)`: обгортка перевіряє `CURRENT_USER["role"]` і, якщо роль інша, піднімає `PermissionError(f"Потрібна роль {role}")`. Не забудь `@wraps`.

In [ ]:
from functools import wraps

CURRENT_USER = {"role": "dispatcher"}        # хто зараз працює — дані з краю (сесія, токен)


def log_call(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"-> {func.__name__}{args}")
        result = func(*args, **kwargs)
        print(f"<- {result}")
        return result
    return wrapper


def requires_role(role):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    def decorator(func):
        @wraps(func)
        def wrapper(*args, **kwargs):
            if CURRENT_USER["role"] != role:
                raise PermissionError(f"Потрібна роль {role}")
            return func(*args, **kwargs)
        return wrapper
    return decorator
    # END SOLUTION


@log_call                         # зовнішній шар
@requires_role("manager")         # внутрішній шар
def change_rate(per_km):
    return {"per_km": per_km}


try:
    change_rate(18)
    raise AssertionError("диспетчер не має змінювати тариф")
except PermissionError as error:
    print("PermissionError:", error)
    assert str(error) == "Потрібна роль manager"

CURRENT_USER["role"] = "manager"
assert change_rate(18) == {"per_km": 18}
assert change_rate.__name__ == "change_rate", "без @wraps ім'я стане wrapper"
CURRENT_USER["role"] = "dispatcher"
print("✅ Завдання 6: два шари, функція всередині не змінювалась")

<details>
<summary>Чому журнал надрукував виклик навіть для диспетчера</summary>

`@log_call` — зовнішній шар: він друкує `->` і лише потім передає виклик усередину, де `requires_role` піднімає помилку. Якщо поміняти шари місцями, у журнал потраплятимуть лише дозволені виклики. Порядок декораторів — теж бізнес-правило, як порядок кроків конвеєра. Без `@` це записується так: `change_rate = log_call(requires_role("manager")(change_rate))`.

</details>

## 🧭 Завдання 7 · Код: збери програму і запусти маршрут

Усі рішення складаються в одну програму: краї на вході й виході, чисте ядро посередині, промокоди з довідника, канал — через гачок.

**Місія:**

1. `ride_steps(ride)` — будує список кроків **для конкретної поїздки**, як `checkout_steps` у розділі 7: нічний коефіцієнт (`make_multiplier`) — лише якщо `is_night(ride["hour"])`; промокод — через `make_promo_step(PROMOS)`;
2. `finish_ride(ride, notify)` — тепер бере кроки з `ride_steps`;
3. прожени `RIDES` і збери прийняті поїздки в `done`, а помилки — в `failed` (через гачок, без `print` у ядрі).

In [ ]:
def ride_steps(ride):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    steps = [validate_ride, km_fare]
    if is_night(ride["hour"]):
        steps.append(make_multiplier(NIGHT_FACTOR))
    steps.append(make_promo_step(PROMOS))
    steps.append(add_pickup)
    steps.append(round_total)
    return steps
    # END SOLUTION


def finish_ride(ride, notify):
    # YOUR CODE HERE
    # BEGIN SOLUTION
    bill = run_pipeline(ride, ride_steps(ride))
    notify(bill)
    return bill
    # END SOLUTION


RIDES = [
    {"km": 12, "hour": 23, "promo": "NIGHT10", "driver": "D-3"},
    {"km": 5, "hour": 14, "promo": None, "driver": "D-1"},
    {"km": 20, "hour": 2, "promo": "FIRST20", "driver": "D-2"},
    {"km": 8, "hour": 10, "promo": "FREE100", "driver": "D-1"},
    {"km": 0, "hour": 12, "promo": None, "driver": "D-2"},
]

done, failed = [], []
for r in RIDES:
    # YOUR CODE HERE
    # BEGIN SOLUTION
    try:
        finish_ride(r, done.append)
    except ValueError as error:
        failed.append(str(error))
    # END SOLUTION

for bill in done:
    notify_driver(bill)                       # край на виході
print("помилки:", failed)

assert [b["total"] for b in done] == [283.0, 115, 400.0]
assert failed == ["невідомий промокод: FREE100", "поїздка без кілометрів"]
assert len(ride_steps(RIDES[1])) == 5 and len(ride_steps(RIDES[0])) == 6, "вдень — без коефіцієнта"
print("✅ Завдання 7: маршрут пройдено")

<details>
<summary>Порахуй сам — перевір відповіді</summary>

- 12 км, 23:00, NIGHT10: 180 × 1.5 = 270 → −10 % = 243 → +40 = **283.0**
- 5 км, 14:00, без промокоду: 75 → +40 = **115**
- 20 км, 02:00, FIRST20: 300 × 1.5 = 450 → −20 % = 360 → +40 = **400.0**
- FREE100 — немає в довіднику; 0 км — не проходить `validate_ride`.

</details>

## 🧭 Кінцева: перенеси на свою систему

Сім питань працюють для будь-якої системи, не лише для доставки й таксі. Обери одну й пройди їх **спершу на папері**, без коду:

- **кав'ярня з собою:** напій, розмір, сироп, картка лояльності, повідомлення бариста;
- **оренда самоката:** хвилини, зона паркування, нічний тариф, штраф, чек у застосунок;
- **власна** — з твого проєкту.

| Питання | Що записати |
|---|---|
| 1. Що входить і що виходить? | чорний ящик: вхід → система → вихід |
| 2. З яких кроків складається перетворення? | список кроків і **чому саме такий порядок** |
| 3. Що обчислює, а що торкається світу? | які функції — ядро, які — край |
| 4. Які правила змінюються без зміни коду? | довідники й коефіцієнти; `partial` чи фабрика |
| 5. Яку поведінку підключати ззовні? | гачки: куди й кому повідомляти |
| 6. Яке правило стосується багатьох функцій? | обгортки: права, журнал, кеш |
| 7. Як усе з'єднати? | `run_pipeline` + функція, що збирає кроки |

Потім напиши код — і перевір, що він повторює схему рядок у рядок.